In [0]:
%pip install faker pandas pydantic mlflow --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import ast

# Reload the data
df = spark.table("default.virtue_foundation_ghana_v_0_3_sheet_1").toPandas()

# Parse list columns safely
def safe_parse(val):
    if val is None or str(val).strip() in ['', 'nan', 'None']:
        return []
    try:
        parsed = ast.literal_eval(str(val))
        return parsed if isinstance(parsed, list) else []
    except:
        return []

df['procedure_list'] = df['procedure'].apply(safe_parse)
df['equipment_list'] = df['equipment'].apply(safe_parse)
df['capability_list'] = df['capability'].apply(safe_parse)
df['specialties_list'] = df['specialties'].apply(safe_parse)

df['needs_idp'] = (
    (df['procedure_list'].apply(len) == 0) |
    (df['equipment_list'].apply(len) == 0)
)

print("Needs IDP parsing:", df['needs_idp'].sum())
print("Has procedures:", (df['procedure_list'].apply(len) > 0).sum())
print("Has equipment:", (df['equipment_list'].apply(len) > 0).sum())
print("Has capabilities:", (df['capability_list'].apply(len) > 0).sum())

# Rename problem column and save
df_clean = df.rename(columns={'mongo DB': 'mongo_db'})
df_clean['procedure_list'] = df_clean['procedure_list'].apply(str)
df_clean['equipment_list'] = df_clean['equipment_list'].apply(str)
df_clean['capability_list'] = df_clean['capability_list'].apply(str)
df_clean['specialties_list'] = df_clean['specialties_list'].apply(str)
df_clean['needs_idp'] = df_clean['needs_idp'].astype(str)

spark.createDataFrame(df_clean).write.mode("overwrite").saveAsTable("default.ghana_facilities_clean")
print("Saved to: default.ghana_facilities_clean")

Needs IDP parsing: 877
Has procedures: 257
Has equipment: 136
Has capabilities: 845
Saved to: default.ghana_facilities_clean
